In [1]:
#installing required libraries
!pip install -q transformers sentencepiece
!pip install -q sentence-transformers
!pip install -q language-tool-python
!pip install -q nltk sacrebleu rouge-score

In [2]:
#importing libraries and checking GPU
import torch
import nltk
import pandas as pd
import numpy as np

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer
import language_tool_python

print("All libraries imported successfully!")

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("We are currently using CPU.")

All libraries imported successfully!
PyTorch version: 2.11.0+cu130
GPU available: True
GPU: Tesla T4


In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "Vamsi/T5_Paraphrase_Paws"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load pretrained T5 model
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Move model to GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("T5 model loaded successfully!")
print("Running on:", device)

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

T5 model loaded successfully!
Running on: cuda


In [4]:
#creating paraphrasing function
def paraphrase_text(text):

    # Add task prefix
    input_text = "paraphrase: " + text + " </s>"

    # Convert text into tokens
    encoding = tokenizer(
        input_text,
        padding="longest",
        max_length=256,
        truncation=True,
        return_tensors="pt"
    )

    # Move tokens to GPU
    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    # Generate paraphrased text
    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=256,
            num_beams=5,
            early_stopping=True
        )

    # Convert generated tokens into text
    paraphrased_text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return paraphrased_text


print("Paraphrasing function created successfully!")

Paraphrasing function created successfully!


In [5]:
original_text = """
Artificial intelligence is transforming the healthcare
industry by improving disease diagnosis and helping
medical professionals provide better patient care.
"""

result = paraphrase_text(original_text)

print("ORIGINAL TEXT:")
print(original_text)

print("\nPARAPHRASED TEXT:")
print(result)

ORIGINAL TEXT:

Artificial intelligence is transforming the healthcare
industry by improving disease diagnosis and helping
medical professionals provide better patient care.


PARAPHRASED TEXT:
Artificial intelligence is transforming the healthcare industry by improving disease diagnosis and helping medical professionals provide better patient care .


In [6]:
#improve paraphrase generation
def paraphrase_text(text):

    input_text = "paraphrase: " + text + " </s>"

    encoding = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=256,
            num_beams=5,
            num_return_sequences=3,
            do_sample=True,
            temperature=0.9,
            top_p=0.95,
            repetition_penalty=1.5,
            no_repeat_ngram_size=2
        )

    paraphrases = [
        tokenizer.decode(output, skip_special_tokens=True)
        for output in outputs
    ]

    return paraphrases

In [7]:
# Testing again
original_text = """
Artificial intelligence is transforming the healthcare
industry by improving disease diagnosis and helping
medical professionals provide better patient care.
"""

results = paraphrase_text(original_text)

print("ORIGINAL TEXT:\n", original_text)

for i, result in enumerate(results, 1):
    print(f"\nPARAPHRASED VERSION {i}:\n{result}")

ORIGINAL TEXT:
 
Artificial intelligence is transforming the healthcare
industry by improving disease diagnosis and helping
medical professionals provide better patient care.


PARAPHRASED VERSION 1:
Artificial intelligence is transforming the healthcare industry by improving disease diagnosis and helping medical professionals provide better patient care .

PARAPHRASED VERSION 2:
Artificial intelligence is transforming the healthcare industry by improving disease diagnosis and helping medical professionals to provide better patient care .

PARAPHRASED VERSION 3:
Artificial intelligence is changing the healthcare industry by improving disease diagnosis and helping medical professionals provide better patient care .


In [8]:
#Load new model
model_name_2 = "humarin/chatgpt_paraphraser_on_T5_base"

tokenizer_2 = AutoTokenizer.from_pretrained(model_name_2)

model_2 = AutoModelForSeq2SeqLM.from_pretrained(model_name_2)

model_2 = model_2.to(device)

print("New paraphrasing model loaded successfully!")

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

New paraphrasing model loaded successfully!


In [9]:
#Test the new model
def improved_paraphrase(text):

    inputs = tokenizer_2(
        "paraphrase: " + text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).input_ids.to(device)

    outputs = model_2.generate(
        inputs,
        num_beams=5,
        num_beam_groups=5,
        num_return_sequences=3,
        repetition_penalty=10.0,
        diversity_penalty=3.0,
        no_repeat_ngram_size=2,
        max_length=128
    )

    results = tokenizer_2.batch_decode(outputs,skip_special_tokens=True)

    return results

In [11]:
# Redefine improved_paraphrase with trust_remote_code=True to fix the group beam search error
def improved_paraphrase(text):
    inputs = tokenizer_2(
        "paraphrase: " + text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).input_ids.to(device)

    outputs = model_2.generate(
        inputs,
        num_beams=5,
        num_beam_groups=5,
        num_return_sequences=3,
        repetition_penalty=10.0,
        diversity_penalty=3.0,
        no_repeat_ngram_size=2,
        max_length=128,
        trust_remote_code=True
    )

    results = tokenizer_2.batch_decode(outputs, skip_special_tokens=True)
    return results

# Generate paraphrases
original_text = """
Artificial intelligence is transforming the healthcare
industry by improving disease diagnosis and helping
medical professionals provide better patient care.
"""

results = improved_paraphrase(original_text)

print("ORIGINAL TEXT:\n", original_text)

for i, result in enumerate(results, 1):
    print(f"\nPARAPHRASED VERSION {i}:\n{result}")

generate.py:   0%|          | 0.00/25.9k [00:00<?, ?B/s]

beam_search.py:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/transformers-community/group-beam-search:
- custom_generate/beam_search.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
[transformers] A new version of the following files was downloaded from https://huggingface.co/transformers-community/group-beam-search:
- custom_generate/generate.py
- beam_search.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
[transformers] Passing `generation_config` together with generation-related arguments=({'num_beams', 'num_return_sequences', 'diversity_penalty', 'no_repeat_ngram_size', 'num_beam_groups', 'repetition_penalty', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation 

ORIGINAL TEXT:
 
Artificial intelligence is transforming the healthcare
industry by improving disease diagnosis and helping
medical professionals provide better patient care.


PARAPHRASED VERSION 1:
The healthcare sector is being impacted by artificial intelligence, which is aiding in the identification of diseases and enhancing patient care.

PARAPHRASED VERSION 2:
The use of artificial intelligence is transforming the healthcare industry by enabling doctors to identify diseases and improve patient care.

PARAPHRASED VERSION 3:
Artificial intelligence is transforming the healthcare industry by enabling doctors to identify diseases and improve patient care.


In [12]:
#saving the model
import pandas as pd

comparison_data = {
    "Original Text": [original_text] * 3,
    "Model": ["T5-PAWS"] * 3,
    "Paraphrased Text": results
}

comparison_df = pd.DataFrame(comparison_data)

comparison_df

,Original Text,Model,Paraphrased Text
0,\nArtificial intelligence is transforming the ...,T5-PAWS,The healthcare sector is being impacted by art...
1,\nArtificial intelligence is transforming the ...,T5-PAWS,The use of artificial intelligence is transfor...
2,\nArtificial intelligence is transforming the ...,T5-PAWS,Artificial intelligence is transforming the he...


In [13]:
# Grammer and spelling test
import language_tool_python

tool = language_tool_python.LanguageTool('en-US')

def grammar_check(text):

    matches = tool.check(text)

    print("Grammar and Spelling Report")
    print("---------------------------")
    print("Total errors:", len(matches))

    for error in matches:
        print("Issue:", error.message)
        print("Suggested correction:", error.replacements[:3])

    if len(matches) == 0:
        print("No grammar or spelling issues found!")

    return matches

In [14]:
for i, sentence in enumerate(results, 1):

    print(f"\nChecking Paraphrase {i}")
    grammar_check(sentence)


Checking Paraphrase 1
Grammar and Spelling Report
---------------------------
Total errors: 0
No grammar or spelling issues found!

Checking Paraphrase 2
Grammar and Spelling Report
---------------------------
Total errors: 0
No grammar or spelling issues found!

Checking Paraphrase 3
Grammar and Spelling Report
---------------------------
Total errors: 0
No grammar or spelling issues found!


In [15]:
# Checking weather the meaning is preserved
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

similarity_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Semantic similarity model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Semantic similarity model loaded successfully!


In [16]:
def calculate_similarity(original, paraphrased):

    embeddings = similarity_model.encode(
        [original, paraphrased]
    )

    score = cosine_similarity(
        [embeddings[0]],
        [embeddings[1]]
    )[0][0]

    return round(float(score), 4)

In [17]:
# testing
for i, sentence in enumerate(results, 1):

    score = calculate_similarity(
        original_text,
        sentence
    )

    print(f"Paraphrase {i}")
    print("Text:", sentence)
    print("Semantic Similarity:", score)
    print("-" * 60)

Paraphrase 1
Text: The healthcare sector is being impacted by artificial intelligence, which is aiding in the identification of diseases and enhancing patient care.
Semantic Similarity: 0.8467
------------------------------------------------------------
Paraphrase 2
Text: The use of artificial intelligence is transforming the healthcare industry by enabling doctors to identify diseases and improve patient care.
Semantic Similarity: 0.9429
------------------------------------------------------------
Paraphrase 3
Text: Artificial intelligence is transforming the healthcare industry by enabling doctors to identify diseases and improve patient care.
Semantic Similarity: 0.9613
------------------------------------------------------------


In [18]:
# BLEU and ROUGE evaluation
import sacrebleu
from rouge_score import rouge_scorer

# Human-written reference paraphrase
reference = """
Artificial intelligence is helping healthcare professionals
diagnose illnesses more effectively and improve the quality
of patient care.
"""

# Initialize ROUGE evaluator
scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=True
)

# Evaluate all generated paraphrases
for i, candidate in enumerate(results, 1):

    # BLEU score
    bleu = sacrebleu.corpus_bleu(
        [candidate],
        [[reference]]
    )

    # ROUGE scores
    rouge = scorer.score(reference, candidate)

    print(f"\nParaphrase {i}")
    print("BLEU Score:", round(bleu.score, 2))
    print("ROUGE-1:", round(rouge['rouge1'].fmeasure, 4))
    print("ROUGE-2:", round(rouge['rouge2'].fmeasure, 4))
    print("ROUGE-L:", round(rouge['rougeL'].fmeasure, 4))
    print("-" * 50)


Paraphrase 1
BLEU Score: 8.07
ROUGE-1: 0.4737
ROUGE-2: 0.1111
ROUGE-L: 0.3684
--------------------------------------------------

Paraphrase 2
BLEU Score: 10.86
ROUGE-1: 0.5405
ROUGE-2: 0.2286
ROUGE-L: 0.4324
--------------------------------------------------

Paraphrase 3
BLEU Score: 16.15
ROUGE-1: 0.5294
ROUGE-2: 0.25
ROUGE-L: 0.4706
--------------------------------------------------


In [19]:
# Originality testing
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def originality_check(original, paraphrased):

    vectorizer = TfidfVectorizer()

    tfidf_matrix = vectorizer.fit_transform(
        [original, paraphrased]
    )

    similarity = cosine_similarity(
        tfidf_matrix[0:1],
        tfidf_matrix[1:2]
    )[0][0]

    lexical_difference = 1 - similarity

    return {
        "Lexical Similarity": round(float(similarity), 4),
        "Lexical Difference": round(float(lexical_difference), 4)
    }

In [20]:
# testing all three
for i, sentence in enumerate(results, 1):

    scores = originality_check(
        original_text,
        sentence
    )

    print(f"\nParaphrase {i}")
    print("Lexical Similarity:", scores["Lexical Similarity"])
    print("Lexical Difference:", scores["Lexical Difference"])
    print("-" * 50)


Paraphrase 1
Lexical Similarity: 0.348
Lexical Difference: 0.652
--------------------------------------------------

Paraphrase 2
Lexical Similarity: 0.4246
Lexical Difference: 0.5754
--------------------------------------------------

Paraphrase 3
Lexical Similarity: 0.4445
Lexical Difference: 0.5555
--------------------------------------------------


In [21]:
#Evaluation table
evaluation_results = []

for i, sentence in enumerate(results, 1):

    # Semantic similarity
    semantic_score = calculate_similarity(
        original_text, sentence
    )

    # Originality indicators
    originality = originality_check(
        original_text, sentence
    )

    # Grammar checking
    grammar_errors = len(tool.check(sentence))

    # BLEU score
    bleu = sacrebleu.corpus_bleu(
        [sentence], [[reference]]
    )

    # ROUGE scores
    rouge = scorer.score(reference, sentence)

    evaluation_results.append({
        "Version": f"Paraphrase {i}",
        "Semantic Similarity": semantic_score,
        "Lexical Similarity": originality["Lexical Similarity"],
        "Lexical Difference": originality["Lexical Difference"],
        "Grammar Errors": grammar_errors,
        "BLEU": round(bleu.score, 2),
        "ROUGE-1": round(rouge["rouge1"].fmeasure, 4),
        "ROUGE-2": round(rouge["rouge2"].fmeasure, 4),
        "ROUGE-L": round(rouge["rougeL"].fmeasure, 4)
    })

evaluation_df = pd.DataFrame(evaluation_results)

evaluation_df

,Version,Semantic Similarity,Lexical Similarity,Lexical Difference,Grammar Errors,BLEU,ROUGE-1,ROUGE-2,ROUGE-L
0,Paraphrase 1,0.8467,0.3480,0.6520,0,8.07,0.4737,0.1111,0.3684
1,Paraphrase 2,0.9429,0.4246,0.5754,0,10.86,0.5405,0.2286,0.4324
2,Paraphrase 3,0.9613,0.4445,0.5555,0,16.15,0.5294,0.2500,0.4706


In [22]:
# Final AI-Powered Paraphrasing Application
def paraphrasing_application():

    print("=" * 60)
    print("       AI-POWERED PARAPHRASING TOOL")
    print("=" * 60)

    # Get input from user
    text = input("\nEnter your text: ")

    if not text.strip():
        print("Please enter some text.")
        return

    # Generate paraphrases
    print("\nGenerating paraphrases...")

    generated_results = improved_paraphrase(text)

    # Display and evaluate results
    for i, sentence in enumerate(generated_results, 1):

        print("\n" + "=" * 60)
        print(f"PARAPHRASE VERSION {i}")
        print("=" * 60)

        print("\nGenerated Text:")
        print(sentence)

        # Semantic similarity
        similarity = calculate_similarity(text, sentence)

        print("\nSemantic Similarity:", similarity)

        # Originality indicators
        originality = originality_check(text, sentence)

        print("Lexical Similarity:",
              originality["Lexical Similarity"])

        print("Lexical Difference:",
              originality["Lexical Difference"])

        # Grammar checking
        grammar_errors = tool.check(sentence)

        print("Grammar Errors:", len(grammar_errors))

    print("\n" + "=" * 60)
    print("PARAPHRASING COMPLETED!")
    print("=" * 60)

In [23]:
paraphrasing_application()

       AI-POWERED PARAPHRASING TOOL

Enter your text: Machine learning is helping businesses analyse large amounts of data, improve decision-making, and automate repetitive tasks.

Generating paraphrases...

PARAPHRASE VERSION 1

Generated Text:
The use of machine learning is enabling businesses to analyze vast amounts of data, improve decision-making, and automate repetitive tasks.

Semantic Similarity: 0.9591
Lexical Similarity: 0.6256
Lexical Difference: 0.3744
Grammar Errors: 0

PARAPHRASE VERSION 2

Generated Text:
Businesses are leveraging machine learning to analyze vast amounts of data, optimize decision-making and automate the execution of repetitive tasks.

Semantic Similarity: 0.9005
Lexical Similarity: 0.5008
Lexical Difference: 0.4992
Grammar Errors: 0

PARAPHRASE VERSION 3

Generated Text:
By analyzing vast amounts of data, businesses are using machine learning to improve decision-making and automate repetitive tasks.

Semantic Similarity: 0.9443
Lexical Similarity: 0.570

In [24]:
#multiple sentences
test_sentences = [
    "Regular exercise improves physical health and reduces stress.",
    "Online education provides students with flexible learning opportunities.",
    "Climate change is affecting agriculture and food production worldwide.",
    "Social media allows people to communicate and share information quickly.",
    "Machine learning helps organizations make better decisions using data."
]

all_test_results = []

for text in test_sentences:

    paraphrases = improved_paraphrase(text)

    for i, sentence in enumerate(paraphrases, 1):

        similarity = calculate_similarity(text, sentence)

        originality = originality_check(text, sentence)

        grammar_errors = len(tool.check(sentence))

        all_test_results.append({
            "Original Text": text,
            "Version": i,
            "Paraphrased Text": sentence,
            "Semantic Similarity": similarity,
            "Lexical Similarity": originality["Lexical Similarity"],
            "Lexical Difference": originality["Lexical Difference"],
            "Grammar Errors": grammar_errors
        })

test_results_df = pd.DataFrame(all_test_results)

print("MULTIPLE SENTENCE EVALUATION")
display(test_results_df)

MULTIPLE SENTENCE EVALUATION


,Original Text,Version,Paraphrased Text,Semantic Similarity,Lexical Similarity,Lexical Difference,Grammar Errors
0,Regular exercise improves physical health and ...,1,Regular exercise benefits both the body and re...,0.9578,0.4211,0.5789,0
1,Regular exercise improves physical health and ...,2,Physical fitness is improved and stress levels...,0.8476,0.3486,0.6514,1
2,Regular exercise improves physical health and ...,3,Performing physical exercises consistently is ...,0.8586,0.3486,0.6514,0
3,Online education provides students with flexib...,1,Online education affords students the flexibil...,0.8448,0.1834,0.8166,0
4,Online education provides students with flexib...,2,Students can learn anything they want by utili...,0.7978,0.2038,0.7962,0
5,Online education provides students with flexib...,3,Flexible learning is a key feature of online e...,0.9025,0.3922,0.6078,0
6,Climate change is affecting agriculture and fo...,1,Climate change is having an impact on agricult...,0.9638,0.5149,0.4851,0
7,Climate change is affecting agriculture and fo...,2,The impact of climate change on agriculture an...,0.9140,0.4635,0.5365,0
8,Climate change is affecting agriculture and fo...,3,Global agriculture and food production are bei...,0.9533,0.4360,0.5640,0
9,Social media allows people to communicate and ...,1,Social media is a platform that facilitates ra...,0.8870,0.2258,0.7742,0


In [25]:
print("OVERALL EVALUATION")

print(
    "Average Semantic Similarity:",
    round(test_results_df["Semantic Similarity"].mean(), 4)
)

print(
    "Average Lexical Difference:",
    round(test_results_df["Lexical Difference"].mean(), 4)
)

print(
    "Total Grammar Errors:",
    test_results_df["Grammar Errors"].sum()
)

print(
    "Total Paraphrases Generated:",
    len(test_results_df)
)

OVERALL EVALUATION
Average Semantic Similarity: 0.879
Average Lexical Difference: 0.6759
Total Grammar Errors: 1
Total Paraphrases Generated: 15


In [26]:
#Identifying grammer error
# Find paraphrases with grammar errors

error_rows = test_results_df[
    test_results_df["Grammar Errors"] > 0
]

for _, row in error_rows.iterrows():

    print("\nOriginal Text:")
    print(row["Original Text"])

    print("\nGenerated Paraphrase:")
    print(row["Paraphrased Text"])

    matches = tool.check(row["Paraphrased Text"])

    print("\nDetected Issues:")

    for error in matches:
        print("Issue:", error.message)
        print("Suggested corrections:", error.replacements[:3])


Original Text:
Regular exercise improves physical health and reduces stress.

Generated Paraphrase:
Physical fitness is improved and stress levels are reduced through regular exercise.

Detected Issues:
Issue: Use a comma before “and” if it connects two independent clauses (unless they are closely connected and short).
Suggested corrections: [', and']


In [27]:
#Final evaluation report
print("FINAL MODEL EVALUATION")
print("-" * 40)

print("Total test sentences:", test_results_df["Original Text"].nunique())
print("Total paraphrases:", len(test_results_df))

print(
    "Average Semantic Similarity:",
    round(test_results_df["Semantic Similarity"].mean(), 4)
)

print(
    "Average Lexical Difference:",
    round(test_results_df["Lexical Difference"].mean(), 4)
)

print(
    "LanguageTool Flags:",
    test_results_df["Grammar Errors"].sum()
)

print("Confirmed Grammar Errors: 0")

FINAL MODEL EVALUATION
----------------------------------------
Total test sentences: 5
Total paraphrases: 15
Average Semantic Similarity: 0.879
Average Lexical Difference: 0.6759
LanguageTool Flags: 1
Confirmed Grammar Errors: 0


In [28]:
final_summary = pd.DataFrame({
    "Evaluation Metric": [
        "Total Test Sentences",
        "Total Paraphrases Generated",
        "Average Semantic Similarity",
        "Average Lexical Difference",
        "LanguageTool Flags",
        "Confirmed Grammar Errors"
    ],
    "Result": [
        5,
        15,
        0.879,
        0.6759,
        1,
        0
    ]
})

display(final_summary)

# Save evaluation files
test_results_df.to_csv(
    "final_paraphrasing_test_results.csv",
    index=False
)

final_summary.to_csv(
    "final_evaluation_summary.csv",
    index=False
)

print("Evaluation files saved successfully!")

,Evaluation Metric,Result
0,Total Test Sentences,5.0000
1,Total Paraphrases Generated,15.0000
2,Average Semantic Similarity,0.8790
3,Average Lexical Difference,0.6759
4,LanguageTool Flags,1.0000
5,Confirmed Grammar Errors,0.0000


Evaluation files saved successfully!


#Conclusion

This project demonstrates the application of transformer-based NLP techniques for automatic text paraphrasing. The system generates multiple alternative sentences and evaluates them using semantic similarity, lexical difference, grammar checking, and reference-based evaluation metrics.

The project provided practical experience in using pre-trained transformer models, text generation, NLP evaluation techniques, and integrating multiple components into a Python application.